# Notebook 01 — Data Cleaning & Exploratory Data Analysis

**Project:** SNAP Participation and Food Access  
**Data:** USDA Food Access Research Atlas (2019)  
**Research question:** Are census tracts with higher SNAP participation more likely to experience low supermarket access, and does this relationship differ between urban and rural areas?

This notebook loads the raw USDA xlsx using `src/clean_food_access.py`, displays key summary statistics, and produces EDA plots saved to `figures/`.

In [ ]:
import sys
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from matplotlib.patches import Patch

sys.path.insert(0, str(Path('..').resolve()))
from src.clean_food_access import run_pipeline, RAW_DIR

FIGURES = Path('..') / 'figures'
FIGURES.mkdir(exist_ok=True)
warnings.filterwarnings('ignore')

## 1. Load and clean data

In [ ]:
df_clean, df_model, dd, xlsx_name = run_pipeline(RAW_DIR, verbose=True)

In [ ]:
print(f'Cleaned dataset : {df_clean.shape[0]:,} rows x {df_clean.shape[1]} columns')
print(f'Modeling dataset: {df_model.shape[0]:,} rows x {df_model.shape[1]} columns')
df_clean.head()

## 2. Key variable summary statistics

In [ ]:
summary_cols = [
    'pct_low_income_low_access', 'pct_snap', 'pct_no_vehicle',
    'PovertyRate', 'MedianFamilyIncome', 'log_median_family_income',
    'pct_children', 'pct_seniors', 'pct_black', 'pct_hispanic', 'pct_asian',
]
df_model[summary_cols].describe().round(2)

### Urban vs. Rural split

In [ ]:
df_model.groupby('urban_label')[['pct_low_income_low_access', 'pct_snap', 'pct_no_vehicle']].describe().round(2)

## 3. Missing value summary for modeling variables

In [ ]:
check_cols = summary_cols + ['Urban', 'urban_label']
missing = df_model[check_cols].isna().sum().rename('missing')
missing_pct = (missing / len(df_model) * 100).rename('pct_missing').round(2)
pd.concat([missing, missing_pct], axis=1)

## 4. Correlation table (numeric modeling variables)

In [ ]:
corr_cols = [
    'pct_low_income_low_access', 'pct_snap', 'pct_no_vehicle',
    'PovertyRate', 'log_median_family_income',
    'pct_children', 'pct_seniors', 'pct_black', 'pct_hispanic', 'pct_asian',
    'Urban',
]
df_model[corr_cols].corr().round(3)

## 5. EDA Plots

All figures are saved to `figures/`.

### Fig 1 — Distribution of the response variable

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(df_model['pct_low_income_low_access'].dropna(), bins=60, edgecolor='white', linewidth=0.3)
ax.set_xlabel('% Low-Income & Low-Access to Supermarkets')
ax.set_ylabel('Number of Census Tracts')
ax.set_title('Distribution of pct_low_income_low_access')
plt.tight_layout()
fig.savefig(FIGURES / '01_hist_response.png', dpi=150)
plt.show()
print('Saved: figures/01_hist_response.png')

### Fig 2 — Distribution of the main explanatory variable

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(df_model['pct_snap'].dropna(), bins=60, edgecolor='white', linewidth=0.3)
ax.set_xlabel('% Occupied Housing Units Receiving SNAP')
ax.set_ylabel('Number of Census Tracts')
ax.set_title('Distribution of pct_snap')
plt.tight_layout()
fig.savefig(FIGURES / '02_hist_pct_snap.png', dpi=150)
plt.show()
print('Saved: figures/02_hist_pct_snap.png')

### Fig 3 — Scatterplot: SNAP participation vs. low-income low-access

In [ ]:
colors = df_model['Urban'].map({1: '#1f77b4', 0: '#d62728'})
legend_elements = [
    Patch(facecolor='#1f77b4', label='Urban'),
    Patch(facecolor='#d62728', label='Rural'),
]

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(
    df_model['pct_snap'], df_model['pct_low_income_low_access'],
    c=colors, alpha=0.25, s=4, linewidths=0
)
ax.legend(handles=legend_elements)
ax.set_xlabel('pct_snap (% HUs with SNAP)')
ax.set_ylabel('pct_low_income_low_access (%)')
ax.set_title('SNAP Participation vs. Low-Income Low-Access (by Urban/Rural)')
plt.tight_layout()
fig.savefig(FIGURES / '03_scatter_snap_vs_lila.png', dpi=150)
plt.show()
print('Saved: figures/03_scatter_snap_vs_lila.png')

### Fig 4 — Boxplot of response by urban/rural

In [ ]:
urban_vals = df_model.loc[df_model['Urban'] == 1, 'pct_low_income_low_access'].dropna()
rural_vals = df_model.loc[df_model['Urban'] == 0, 'pct_low_income_low_access'].dropna()

fig, ax = plt.subplots(figsize=(5, 5))
ax.boxplot(
    [urban_vals, rural_vals], labels=['Urban', 'Rural'], patch_artist=True,
    boxprops=dict(facecolor='#aec7e8'),
    medianprops=dict(color='black', linewidth=2),
    flierprops=dict(marker='.', markersize=2, alpha=0.3)
)
ax.set_ylabel('pct_low_income_low_access (%)')
ax.set_title('Low-Income Low-Access by Urban/Rural')
plt.tight_layout()
fig.savefig(FIGURES / '04_boxplot_lila_by_urban.png', dpi=150)
plt.show()
print('Saved: figures/04_boxplot_lila_by_urban.png')

### Fig 5 — Scatterplot: No-vehicle households vs. low-income low-access

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(
    df_model['pct_no_vehicle'], df_model['pct_low_income_low_access'],
    c=colors, alpha=0.25, s=4, linewidths=0
)
ax.legend(handles=legend_elements)
ax.set_xlabel('pct_no_vehicle (% HUs without a vehicle)')
ax.set_ylabel('pct_low_income_low_access (%)')
ax.set_title('Vehicle Access vs. Low-Income Low-Access (by Urban/Rural)')
plt.tight_layout()
fig.savefig(FIGURES / '05_scatter_noveh_vs_lila.png', dpi=150)
plt.show()
print('Saved: figures/05_scatter_noveh_vs_lila.png')

## 6. Output file locations

| File | Description |
|---|---|
| `data/cleaned/food_access_cleaned.csv` | Full cleaned dataset (72,531 tracts, 159 columns) |
| `data/cleaned/food_access_modeling.csv` | Regression-ready subset (42,156 tracts, 26 columns) |
| `data/cleaned/data_dictionary_cleaned.csv` | Variable lookup from USDA |
| `report/data_cleaning_summary.md` | Cleaning log and regression notes |
| `figures/` | EDA plots |

See `report/data_cleaning_summary.md` for the suggested regression models and notes on handling spatial clustering.